<a href="https://colab.research.google.com/github/zhisenj241-ship-it/Solar-Flare-Prediction-code/blob/main/Solar_Flare.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys
import subprocess
import importlib.util

AUTO_INSTALL_MISSING_PACKAGES = True

required_packages = {
    "drms": "drms",
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "tqdm": "tqdm",
    "sunpy": "sunpy",
    "zeep": "zeep"
}

for package_name, import_name in required_packages.items():
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing missing package: {package_name}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])
    else:
        print(f"Package already available: {package_name}")
from sunpy.net import Fido
from sunpy.net import attrs as a
import drms
import random
import numpy as np
from datetime import datetime, timedelta
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, ConfusionMatrixDisplay
from sklearn.feature_selection import SelectKBest, f_classif

Installing missing package: drms
Package already available: pandas
Package already available: numpy
Package already available: matplotlib
Package already available: tqdm
Installing missing package: sunpy
Installing missing package: zeep


In [ ]:

#Queries registed flare events for SHARP data

event_type = "FL"
tstart = "2010/5/1"
tend = "2014/5/1"
result = Fido.search(a.Time(tstart, tend),
                     a.hek.EventType(event_type),
                     a.hek.FL.GOESCls > "M1.0",
                     a.hek.OBS.Observatory == "GOES")
hek_results = result["hek"]
filtered_results = hek_results["event_starttime", "event_peaktime",
                               "event_endtime", "fl_goescls", "ar_noaanum","hrc_coord","hpc_coord"]
print(len(filtered_results))

417


In [ ]:
import drms
import pandas as pd
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed

client = drms.Client()

hrs_before = 24

#filter valid flares
valid_flares = []
for flare in filtered_results:
    noaa_num = flare.get("ar_noaanum")
    flare_time_str = str(flare.get("event_starttime", ""))

    if not noaa_num or noaa_num == 0 or str(noaa_num).upper() == "NONE":
        continue

    flare_dt = datetime.fromisoformat(flare_time_str.replace("Z", ""))

    prior_24h_dt = flare_dt - timedelta(hours=hrs_before)
    jsoc_time_str = prior_24h_dt.strftime("%Y.%m.%d_%H:%M:%S_TAI")

    valid_flares.append({
        "noaa_num": str(noaa_num),
        "jsoc_time_str": jsoc_time_str,
        "flare_starttime": flare_time_str,
        "flare_class": flare.get("fl_goescls")
    })
KEYWORDS = [
    "HARPNUM",
    "NOAA_ARS",
    "T_REC",
    "USFLUX",
    "TOTPOT",
    "TOTUSJZ",
    "SAVNCPP",
    "AREA_ACR",
    "MEANPOT",
    "R_VALUE",
    "TOTUSJH",
]

def fetch_single_sharp(flare_item):
    noaa_num = flare_item["noaa_num"]
    jsoc_time_str = flare_item["jsoc_time_str"]
    query_spec = f"hmi.sharp_720s[][{jsoc_time_str}]"

    try:
        keys = client.query(query_spec, key=KEYWORDS)
        if not keys.empty:
            matched = keys[keys["NOAA_ARS"].astype(str).str.contains(noaa_num)]
            if not matched.empty:
                record = matched.iloc[0].to_dict()
                record["FLARE_LABEL"] = 1
                record["FLARE_NOAA_NUM"] = noaa_num
                record["FLARE_STARTTIME"] = flare_item["flare_starttime"]
                record["FLARE_CLASS"] = flare_item["flare_class"]
                return record
    except Exception as e:
        print(f"Error querying NOAA {noaa_num} at {jsoc_time_str}: {e}")
    return None

sharp_data_list = []
max_workers = 10  # JSOC supports  10 concurrent requests

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = [executor.submit(fetch_single_sharp, flare) for flare in valid_flares]
    for future in as_completed(futures):
        result = future.result()
        if result is not None:
            sharp_data_list.append(result)

#Convert  to DataFrame
found_sharp_data = pd.DataFrame(sharp_data_list)
print(f"Successfully matched {len(found_sharp_data)} / {len(filtered_results)} flares to SHARP data.")
sample_size = min(200, len(found_sharp_data))
sharp_positive_classes = found_sharp_data.sample(n=sample_size) if sample_size > 0 else pd.DataFrame()

Successfully matched 378 / 417 flares to SHARP data.


In [ ]:
#Queries random flare events that don'y overlap with flare events


buffer_hrs = 24 #changes how many hrs before
flare_exclusion_windows = []

for flare in hek_results:
    start_str = str(flare["event_starttime"]).replace("Z", "").replace(" ", "T")
    end_str = str(flare["event_endtime"]).replace("Z", "").replace(" ", "T")

    t_start = datetime.fromisoformat(start_str)
    t_end = datetime.fromisoformat(end_str)

    # Expand interval by 24 hours on both sides
    excl_start = t_start - timedelta(hours=buffer_hrs)
    excl_end = t_end + timedelta(hours=buffer_hrs)
    #adds all flare into one giant exclusion window
    flare_exclusion_windows.append((excl_start, excl_end))

print(f"Loaded {len(flare_exclusion_windows)} flare exclusion windows.")

def is_near_flare(candidate_dt):
    for window_start, window_end in flare_exclusion_windows:
        if window_start <= candidate_dt <= window_end:
            return True
    return False

client = drms.Client()
non_flare_samples = []

start_date = datetime(2010, 5, 1)
end_date = datetime(2014, 5, 1)
total_seconds = int((end_date - start_date).total_seconds())

#Defaults to 500 non-flare events but can change to adjust class ratioimport drms
import random
import pandas as pd
from datetime import datetime, timedelta
from bisect import bisect_left
from concurrent.futures import ThreadPoolExecutor, as_completed

#Build and merge exclusion windows
buffer_hrs = 24
raw_windows = []

for flare in hek_results:
    start_str = str(flare["event_starttime"]).replace("Z", "").replace(" ", "T")
    end_str = str(flare["event_endtime"]).replace("Z", "").replace(" ", "T")

    t_start = datetime.fromisoformat(start_str)
    t_end = datetime.fromisoformat(end_str)

    raw_windows.append((t_start - timedelta(hours=buffer_hrs), t_end + timedelta(hours=buffer_hrs)))

raw_windows.sort(key=lambda x: x[0])
merged_windows = []
for start, end in raw_windows:
    if not merged_windows or merged_windows[-1][1] < start:
        merged_windows.append((start, end))
    else:
        merged_windows[-1] = (merged_windows[-1][0], max(merged_windows[-1][1], end))

window_starts = [w[0] for w in merged_windows]

def is_near_flare_fast(candidate_dt):
    idx = bisect_left(window_starts, candidate_dt)
    if idx > 0:
        prev_start, prev_end = merged_windows[idx - 1]
        if prev_start <= candidate_dt <= prev_end:
            return True

    if idx < len(merged_windows):
        next_start, next_end = merged_windows[idx]
        if next_start <= candidate_dt <= next_end:
            return True

    return False

start_date = datetime(2010, 5, 1)
end_date = datetime(2014, 5, 1)
total_seconds = int((end_date - start_date).total_seconds())

target_samples = 500
candidate_timestamps = []

while len(candidate_timestamps) < target_samples * 2:
    random_sec = random.randint(0, total_seconds)
    candidate_dt = start_date + timedelta(seconds=random_sec)

    if not is_near_flare_fast(candidate_dt):
        candidate_timestamps.append(candidate_dt.strftime("%Y.%m.%d_%H:%M:%S_TAI"))

KEYWORDS = [
    "HARPNUM",
    "NOAA_ARS",
    "T_REC",
    "USFLUX",
    "TOTPOT",
    "TOTUSJZ",
    "SAVNCPP",
    "AREA_ACR",
    "MEANPOT",
    "R_VALUE",
    "TOTUSJH",
]

client = drms.Client()

def fetch_negative_sample(jsoc_time_str):
    query_spec = f"hmi.sharp_720s[][{jsoc_time_str}/12m]"
    try:
        keys = client.query(query_spec, key=KEYWORDS)
        if keys is not None and not keys.empty:
            sample = keys.sample(n=1).iloc[0].to_dict()
            sample["FLARE_LABEL"] = 0
            sample["QUERY_TIMESTAMP"] = jsoc_time_str
            return sample
    except Exception:
        pass
    return None

non_flare_samples = []
max_workers = 10

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = [executor.submit(fetch_negative_sample, ts) for ts in candidate_timestamps]

    for future in as_completed(futures):
        res = future.result()
        if res is not None:
            non_flare_samples.append(res)
            if len(non_flare_samples) >= target_samples:
                executor.shutdown(wait=False, cancel_futures=True)
                break

sharp_negative_classes = pd.DataFrame(non_flare_samples[:target_samples])
print(f"Sampling Complete! Collected {len(sharp_negative_classes)} non-flare samples.")
while len(non_flare_samples) < 500:
    random_sec = random.randint(0, total_seconds)
    candidate_dt = start_date + timedelta(seconds=random_sec)
    #Makes sure selected time doesn't coincide with flare event
    if is_near_flare(candidate_dt):
        continue
    jsoc_time_str = candidate_dt.strftime("%Y.%m.%d_%H:%M:%S_TAI")

    try:
        keys = client.query(
            f"hmi.sharp_720s[][{jsoc_time_str}]",
            key = '**',
            )
            # key="HARPNUM, NOAA_ARS, T_REC, USFLUX,TOTBSQ,TOTPOT,TOTUSJZ,SAVNCPP,AREA_ACR,TOTFZ,MEANPOT,R_VALUE,EPSZ,SHRGT45,MEANSHR,MEANGAM,MEANGBT,MEANGBZ",


        if not keys.empty:
            sample = keys.sample(n=1).iloc[0].to_dict()
            sample["FLARE_LABEL"] = 0  # Non-flare label
            sample["QUERY_TIMESTAMP"] = jsoc_time_str

            non_flare_samples.append(sample)

    except Exception as e:
        continue

sharp_negative_classes = pd.DataFrame(non_flare_samples)
print("\nSampling Complete!")

Loaded 417 flare exclusion windows.
Sampling Complete! Collected 500 non-flare samples.

Sampling Complete!


In [ ]:
#Queries registed flare events for SMARP data

event_type = "FL"
tstart = "2000/1/1"
tend = "2004/1/1"
result = Fido.search(a.Time(tstart, tend),
                     a.hek.EventType(event_type),
                     a.hek.FL.GOESCls > "M1.0",
                     a.hek.OBS.Observatory == "GOES")
hek_results = result["hek"]
filtered_results = hek_results["event_starttime", "event_peaktime",
                               "event_endtime", "fl_goescls", "ar_noaanum","hrc_coord","hpc_coord"]
print(len(filtered_results))

862


In [ ]:
import drms
import pandas as pd
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed

client = drms.Client()
valid_flares = []
for flare in filtered_results:
    noaa_num = flare.get("ar_noaanum")
    flare_time_str = str(flare.get("event_starttime", ""))

    if not noaa_num or noaa_num == 0 or str(noaa_num).upper() in ["NONE", "MASKED", "NAN"]:
        continue

    flare_dt = datetime.fromisoformat(flare_time_str.replace("Z", ""))
    prior_24h_dt = flare_dt - timedelta(hours=hrs_before)
    jsoc_time_str = prior_24h_dt.strftime("%Y.%m.%d_%H:%M:%S_TAI")

    valid_flares.append({
        "noaa_num": str(noaa_num).strip(),
        "jsoc_time_str": jsoc_time_str,
        "flare_starttime": flare_time_str,
        "flare_class": flare.get("fl_goescls")
    })

# SMARP specific keywords
KEYWORDS = [
    "TARPNUM", "NOAA_ARS", "T_REC",
    "AREA_SPT",
    "AREA_ACR",
    "NPIX_SPT",
    "USFLUXL",
    "R_VALUE",
    "SIZE_ACR",
    "MTOT",
    "MNEG_TOT",
    "MPOS_TOT",
    "MSTDEV",]

def fetch_single_smarp(flare_item):
    raw_noaa = flare_item["noaa_num"]
    jsoc_time_str = flare_item["jsoc_time_str"]

    noaa_variants = [raw_noaa]
    if len(raw_noaa) == 5 and raw_noaa.startswith("1"):
        noaa_variants.append(raw_noaa[1:])
    elif len(raw_noaa) <= 4:
        noaa_variants.append(f"1{raw_noaa.zfill(4)}")

    pattern = r"\b(?:" + "|".join(noaa_variants) + r")\b"

    # Use 96m time range window for SMARP cadence
    query_spec = f"mdi.smarp_96m[][{jsoc_time_str}/96m]"

    try:
        keys = client.query(query_spec, key=KEYWORDS)
        if keys is not None and not keys.empty:
            matched = keys[keys["NOAA_ARS"].astype(str).str.contains(pattern, regex=True, na=False)]

            if not matched.empty:
                record = matched.iloc[0].to_dict()
                record["FLARE_LABEL"] = 1
                record["FLARE_NOAA_NUM"] = raw_noaa
                record["FLARE_STARTTIME"] = flare_item["flare_starttime"]
                record["FLARE_CLASS"] = flare_item["flare_class"]
                return record
    except Exception as e:
        print(f"Error querying NOAA {raw_noaa} at {jsoc_time_str}: {e}")
    return None
smarp_data_list = []
max_workers = 10

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = [executor.submit(fetch_single_smarp, flare) for flare in valid_flares]
    for future in as_completed(futures):
        res = future.result()
        if res is not None:
            smarp_data_list.append(res)

found_smarp_data = pd.DataFrame(smarp_data_list)
print(f"Successfully matched {len(found_smarp_data)} / {len(valid_flares)} valid flares to SMARP data.")

sample_size = min(200, len(found_smarp_data))
smarp_positive_classes = found_smarp_data.sample(n=sample_size) if sample_size > 0 else pd.DataFrame()

Successfully matched 424 / 434 valid flares to SMARP data.


In [ ]:
import drms
import random
import pandas as pd
from datetime import datetime, timedelta
from bisect import bisect_left
from concurrent.futures import ThreadPoolExecutor, as_completed

#Build and merge exclusion windows
buffer_hrs = 24
raw_windows = []

for flare in hek_results:
    start_str = str(flare["event_starttime"]).replace("Z", "").replace(" ", "T")
    end_str = str(flare["event_endtime"]).replace("Z", "").replace(" ", "T")

    t_start = datetime.fromisoformat(start_str)
    t_end = datetime.fromisoformat(end_str)

    raw_windows.append((t_start - timedelta(hours=buffer_hrs), t_end + timedelta(hours=buffer_hrs)))

raw_windows.sort(key=lambda x: x[0])

merged_windows = []
for start, end in raw_windows:
    if not merged_windows or merged_windows[-1][1] < start:
        merged_windows.append((start, end))
    else:
        merged_windows[-1] = (merged_windows[-1][0], max(merged_windows[-1][1], end))

window_starts = [w[0] for w in merged_windows]

def is_near_flare_fast(candidate_dt):
    idx = bisect_left(window_starts, candidate_dt)

    if idx > 0:
        prev_start, prev_end = merged_windows[idx - 1]
        if prev_start <= candidate_dt <= prev_end:
            return True

    if idx < len(merged_windows):
        next_start, next_end = merged_windows[idx]
        if next_start <= candidate_dt <= next_end:
            return True

    return False

start_date = datetime(2000, 1, 1)
end_date = datetime(2004, 1, 1)
total_seconds = int((end_date - start_date).total_seconds())

target_samples = 500
candidate_timestamps = []

while len(candidate_timestamps) < target_samples * 2:
    random_sec = random.randint(0, total_seconds)
    candidate_dt = start_date + timedelta(seconds=random_sec)

    if not is_near_flare_fast(candidate_dt):
        candidate_timestamps.append(candidate_dt.strftime("%Y.%m.%d_%H:%M:%S_TAI"))

KEYWORDS = ["TARPNUM", "NOAA_ARS", "T_REC",
    "AREA_SPT",
    "AREA_ACR",
    "NPIX_SPT",
    "USFLUXL",
    "R_VALUE",
    "SIZE_ACR",
    "MTOT",
    "MNEG_TOT",
    "MPOS_TOT",
    "MSTDEV",]

client = drms.Client()

def fetch_smarp_negative_sample(jsoc_time_str):
    query_spec = f"mdi.smarp_96m[][{jsoc_time_str}/96m]"
    try:
        keys = client.query(query_spec, key=KEYWORDS)
        if keys is not None and not keys.empty:
            sample = keys.sample(n=1).iloc[0].to_dict()
            sample["FLARE_LABEL"] = 0
            sample["QUERY_TIMESTAMP"] = jsoc_time_str
            return sample
    except Exception:
        pass
    return None

non_flare_samples = []
max_workers = 10

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = [executor.submit(fetch_smarp_negative_sample, ts) for ts in candidate_timestamps]

    for future in as_completed(futures):
        res = future.result()
        if res is not None:
            non_flare_samples.append(res)
            if len(non_flare_samples) >= target_samples:
                executor.shutdown(wait=False, cancel_futures=True)
                break

smarp_negative_classes = pd.DataFrame(non_flare_samples[:target_samples])
print(f"Sampling Complete! Collected {len(smarp_negative_classes)} SMARP non-flare samples.")

Sampling Complete! Collected 500 SMARP non-flare samples.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import GroupKFold, GroupShuffleSplit

def prepare_train_val(X_tr, X_va):
    """Preprocesses Train and Validation/Test sets using statistics computed

    STRICTLY from the Training set to prevent data leakage.
    """
    X_tr = X_tr.apply(pd.to_numeric, errors="coerce").replace(
        [np.inf, -np.inf], np.nan
    )
    X_va = X_va.apply(pd.to_numeric, errors="coerce").replace(
        [np.inf, -np.inf], np.nan
    )

    valid_cols = X_tr.columns[X_tr.nunique() > 1]
    X_tr = X_tr[valid_cols]
    X_va = X_va[valid_cols]

    # change NaNs using only training set medians
    tr_medians = X_tr.median()
    X_tr = X_tr.fillna(tr_medians)
    X_va = X_va.fillna(tr_medians)

    return X_tr, X_va


def run_group_kfold_pipeline(
    df,
    group_col,
    target_col,
    feature_cols,
    n_splits=5,
    k_features=12,
    test_size=0.15,
):
    df_clean = df.dropna(subset=[target_col, group_col]).copy()
    X = df_clean[feature_cols]
    y = df_clean[target_col].astype(int)
    groups = df_clean[group_col]
    gss_test = GroupShuffleSplit(
        n_splits=1, test_size=test_size, random_state=42
    )
    train_val_idx, test_idx = next(gss_test.split(X, y, groups=groups))

    X_train_val = X.iloc[train_val_idx].reset_index(drop=True)
    y_train_val = y.iloc[train_val_idx].reset_index(drop=True)
    groups_train_val = groups.iloc[train_val_idx].reset_index(drop=True)

    X_test_raw = X.iloc[test_idx].reset_index(drop=True)
    y_test = y.iloc[test_idx].reset_index(drop=True)

    gkf = GroupKFold(n_splits=n_splits)

    fold_accuracies = []
    fold_tss_scores = []
    best_thresholds = []


    for fold, (tr_idx, va_idx) in enumerate(
        gkf.split(X_train_val, y_train_val, groups=groups_train_val), start=1
    ):
        X_tr_fold = X_train_val.iloc[tr_idx]
        y_tr_fold = y_train_val.iloc[tr_idx]
        X_va_fold = X_train_val.iloc[va_idx]
        y_va_fold = y_train_val.iloc[va_idx]

        X_tr_fold, X_va_fold = prepare_train_val(X_tr_fold, X_va_fold)

        selector = SelectKBest(
            score_func=f_classif, k=min(k_features, X_tr_fold.shape[1])
        )
        selector.fit(X_tr_fold, y_tr_fold)

        selected_cols = X_tr_fold.columns[selector.get_support()]
        X_tr_selected = X_tr_fold[selected_cols]
        X_va_selected = X_va_fold[selected_cols]

        #fit Random Forest
        rf = RandomForestClassifier(
            n_estimators=100, class_weight="balanced", random_state=42 + fold
        )
        rf.fit(X_tr_selected, y_tr_fold)

        val_probs = rf.predict_proba(X_va_selected)[:, 1]
        best_fold_thresh = 0.5
        best_fold_tss = -1.0
        best_fold_acc = 0.0

        for thresh in np.arange(0.1, 0.9, 0.05):
            val_preds = (val_probs >= thresh).astype(int)
            cm_v = confusion_matrix(y_va_fold, val_preds)

            if cm_v.shape == (2, 2):
                tn, fp, fn, tp = cm_v.ravel()
                tpr_v = tp / (tp + fn) if (tp + fn) > 0 else 0
                fpr_v = fp / (fp + tn) if (fp + tn) > 0 else 0
                tss_v = tpr_v - fpr_v

                if tss_v > best_fold_tss:
                    best_fold_tss = tss_v
                    best_fold_thresh = thresh
                    best_fold_acc = accuracy_score(y_va_fold, val_preds)

        fold_accuracies.append(best_fold_acc)
        fold_tss_scores.append(best_fold_tss)
        best_thresholds.append(best_fold_thresh)

        print(
            f"Fold {fold}/{n_splits} | Validation Accuracy: {best_fold_acc:.4f} | "
            f"Validation TSS: {best_fold_tss:.4f} (Optimal Thresh: {best_fold_thresh:.2f})"
        )
    cv_mean_acc = np.mean(fold_accuracies)
    cv_std_acc = np.std(fold_accuracies)
    cv_mean_tss = np.mean(fold_tss_scores)
    cv_std_tss = np.std(fold_tss_scores)
    mean_optimal_threshold = np.mean(best_thresholds)

    print(f"  Mean Validation Accuracy: {cv_mean_acc:.4f} ± {cv_std_acc:.4f}")
    print(f"  Mean Validation TSS:      {cv_mean_tss:.4f} ± {cv_std_tss:.4f}")
    print(f"  Average Optimal Threshold: {mean_optimal_threshold:.2f}")

    X_train_val_clean, X_test_clean = prepare_train_val(
        X_train_val, X_test_raw
    )

    selector_final = SelectKBest(
        score_func=f_classif, k=min(k_features, X_train_val_clean.shape[1])
    )
    selector_final.fit(X_train_val_clean, y_train_val)

    final_cols = X_train_val_clean.columns[selector_final.get_support()]
    X_tr_val_final = X_train_val_clean[final_cols]
    X_te_final = X_test_clean[final_cols]

    rf_final = RandomForestClassifier(
        n_estimators=100, class_weight="balanced", random_state=42
    )
    rf_final.fit(X_tr_val_final, y_train_val)

    test_probs = rf_final.predict_proba(X_te_final)[:, 1]
    test_preds = (test_probs >= mean_optimal_threshold).astype(int)

    cm_t = confusion_matrix(y_test, test_preds)
    tn, fp, fn, tp = cm_t.ravel()
    tpr_t = tp / (tp + fn) if (tp + fn) > 0 else 0
    fpr_t = fp / (fp + tn) if (fp + tn) > 0 else 0
    tss_t = tpr_t - fpr_t

    print(f"  Test Accuracy: {accuracy_score(y_test, test_preds):.4f}")
    print(f"  Test TPR (Recall): {tpr_t:.4f}")
    print(f"  Test FPR:          {fpr_t:.4f}")
    print(f"  Test TSS:          {tss_t:.4f}")
    print(f"  Selected Features: {list(final_cols)}")

sharp_dataset = pd.concat(
    [
        sharp_positive_classes,
        sharp_negative_classes.sample(n=500, random_state=42),
    ],
    ignore_index=True,
)

sharp_features = [
    "HARPNUM",
    "NOAA_ARS",
    "T_REC",
    "USFLUX",
    "TOTPOT",
    "TOTUSJZ",
    "SAVNCPP",
    "AREA_ACR",
    "MEANPOT",
    "R_VALUE",
    "TOTUSJH",
]
run_group_kfold_pipeline(
    df=sharp_dataset,
    group_col="HARPNUM",
    target_col="FLARE_LABEL",
    feature_cols=sharp_features,
    n_splits=5,
    k_features=12,
    test_size=0.15,
)
smarp_dataset = pd.concat(
    [
        smarp_positive_classes,
        smarp_negative_classes.sample(n=500, random_state=42),
    ],
    ignore_index=True,
)

smarp_features = ["TARPNUM", "NOAA_ARS", "T_REC",
    "AREA_SPT",
    "AREA_ACR",
    "NPIX_SPT",
    "USFLUXL",
    "R_VALUE",
    "SIZE_ACR",
    "MTOT",
    "MNEG_TOT",
    "MPOS_TOT",
    "MSTDEV",]
run_group_kfold_pipeline(
    df=smarp_dataset,
    group_col="TARPNUM",
    target_col="FLARE_LABEL",
    feature_cols=smarp_features,
    n_splits=5,
    k_features=12,
    test_size=0.15,
)

--- PROCESSING SHARP DATASET WITH GROUP K-FOLD CV ---

--- Running 5-Fold Group CV (Grouping by HARPNUM) ---
Fold 1/5 | Validation Accuracy: 0.9339 | Validation TSS: 0.8585 (Optimal Thresh: 0.30)
Fold 2/5 | Validation Accuracy: 0.9339 | Validation TSS: 0.8501 (Optimal Thresh: 0.50)
Fold 3/5 | Validation Accuracy: 0.8926 | Validation TSS: 0.8115 (Optimal Thresh: 0.25)
Fold 4/5 | Validation Accuracy: 0.9167 | Validation TSS: 0.8016 (Optimal Thresh: 0.20)
Fold 5/5 | Validation Accuracy: 0.9083 | Validation TSS: 0.7983 (Optimal Thresh: 0.25)

--------------------------------------------------
GROUP K-FOLD CV SUMMARY (5 Folds):
  Mean Validation Accuracy: 0.9171 ± 0.0158
  Mean Validation TSS:      0.8240 ± 0.0252
  Average Optimal Threshold: 0.30
--------------------------------------------------

UNTOUCHED TEST SET RESULTS (Evaluated at Threshold = 0.30):
  Test Accuracy: 0.9278
  Test TPR (Recall): 0.9600
  Test FPR:          0.0833
  Test TSS:          0.8767
  Selected Features: ['HARP

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import GroupShuffleSplit


# =====================================================================
# 1. LEAKAGE-SAFE PREPROCESSING FUNCTION
# =====================================================================
def prepare_train_val_test(X_tr, X_va, X_te):
    """Preprocesses Train, Validation, and Test sets using statistics computed

    STRICTLY from the Training set to prevent data leakage.
    """
    X_tr = X_tr.apply(pd.to_numeric, errors="coerce").replace(
        [np.inf, -np.inf], np.nan
    )
    X_va = X_va.apply(pd.to_numeric, errors="coerce").replace(
        [np.inf, -np.inf], np.nan
    )
    X_te = X_te.apply(pd.to_numeric, errors="coerce").replace(
        [np.inf, -np.inf], np.nan
    )

    # 1. Drop constant columns based ONLY on training data
    valid_cols = X_tr.columns[X_tr.nunique() > 1]
    X_tr = X_tr[valid_cols]
    X_va = X_va[valid_cols]
    X_te = X_te[valid_cols]

    # 2. Impute NaNs using ONLY Training set medians
    tr_medians = X_tr.median()
    X_tr = X_tr.fillna(tr_medians)
    X_va = X_va.fillna(tr_medians)
    X_te = X_te.fillna(tr_medians)

    return X_tr, X_va, X_te


# =====================================================================
# 2. HIERARCHICAL GROUP SPLIT FUNCTION (Train / Validation / Test)
# =====================================================================
def group_train_val_test_split(
    df, group_col, target_col, feature_cols, test_size=0.15, val_size=0.15
):
    """Splits a dataframe into Train, Validation, and Test sets based on Active Region groups

    to prevent spatial/temporal leakage.
    """
    df_clean = df.dropna(subset=[target_col, group_col]).copy()
    X = df_clean[feature_cols]
    y = df_clean[target_col].astype(int)
    groups = df_clean[group_col]

    # --- Step A: Split into (Train + Val) and Test ---
    gss_test = GroupShuffleSplit(
        n_splits=1, test_size=test_size, random_state=42
    )
    train_val_idx, test_idx = next(gss_test.split(X, y, groups=groups))

    X_train_val, y_train_val = X.iloc[train_val_idx], y.iloc[train_val_idx]
    groups_train_val = groups.iloc[train_val_idx]

    X_test, y_test = X.iloc[test_idx], y.iloc[test_idx]

    # --- Step B: Split (Train + Val) into Train and Validation ---
    val_relative_size = val_size / (1.0 - test_size)
    gss_val = GroupShuffleSplit(
        n_splits=1, test_size=val_relative_size, random_state=42
    )
    train_idx, val_idx = next(
        gss_val.split(X_train_val, y_train_val, groups=groups_train_val)
    )

    X_train, y_train = (
        X_train_val.iloc[train_idx],
        y_train_val.iloc[train_idx],
    )
    X_val, y_val = X_train_val.iloc[val_idx], y_train_val.iloc[val_idx]

    return X_train, X_val, X_test, y_train, y_val, y_test


# =====================================================================
# 3. FEATURE COUNT EXPERIMENT & EVALUATION FUNCTION
# =====================================================================
def run_feature_count_experiments(
    X_tr, X_va, X_te, y_tr, y_va, y_te, dataset_name="SHARP", n_runs=10
):
    print(f"==================================================")
    print(f"  FEATURE COUNT EXPERIMENTS FOR {dataset_name} DATASET")
    print(f"==================================================")

    feature_counts = [8, 10]
    val_summary = []

    best_feat_count = None
    highest_val_tss = -1.0

    for k in feature_counts:
        # Rank & select top k features strictly on Training Set using ANOVA F-value (Fisher score)
        selector = SelectKBest(score_func=f_classif, k=min(k, X_tr.shape[1]))
        selector.fit(X_tr, y_tr)

        # Retrieve selected feature names and their corresponding F-scores
        selected_mask = selector.get_support()
        selected_cols = X_tr.columns[selected_mask]
        selected_scores = selector.scores_[selected_mask]

        # Create a sorted DataFrame of selected features and scores
        feature_scores_df = (
            pd.DataFrame(
                {"Feature": selected_cols, "F_Score": selected_scores}
            )
            .sort_values(by="F_Score", ascending=False)
            .reset_index(drop=True)
        )

        # Print top k features and their F-scores
        print(f"\n--- Top {k} Selected Features (ANOVA F-Score) ---")
        for idx, row in feature_scores_df.iterrows():
            print(f"  {idx + 1}. {row['Feature']}: {row['F_Score']:.4f}")

        X_tr_k = X_tr[selected_cols]
        X_va_k = X_va[selected_cols]

        val_accs = []
        val_tss_scores = []

        for seed in range(n_runs):
            rf = RandomForestClassifier(
                n_estimators=100,
                class_weight="balanced",
                random_state=42 + seed,
            )
            rf.fit(X_tr_k, y_tr)

            # Predict on Validation Set
            y_val_pred = rf.predict(X_va_k)

            acc = accuracy_score(y_va, y_val_pred)

            cm = confusion_matrix(y_va, y_val_pred)
            if cm.shape == (2, 2):
                tn, fp, fn, tp = cm.ravel()
                tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
                fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
                tss = tpr - fpr
            else:
                tss = 0.0

            val_accs.append(acc)
            val_tss_scores.append(tss)

        mean_acc = np.mean(val_accs)
        std_acc = np.std(val_accs)
        mean_tss = np.mean(val_tss_scores)
        std_tss = np.std(val_tss_scores)

        val_summary.append({
            "k_features": k,
            "mean_val_acc": mean_acc,
            "std_val_acc": std_acc,
            "mean_val_tss": mean_tss,
            "std_val_tss": std_tss,
            "selected_features": list(selected_cols),
        })

        print(
            f"\n[Top {k} Features] Validation Performance over {n_runs} runs:"
        )
        print(f"  Mean Accuracy: {mean_acc:.4f} ± {std_acc:.4f}")
        print(f"  Mean TSS:      {mean_tss:.4f} ± {std_tss:.4f}")
        print(f"  Selected Features: {list(selected_cols)}")

        if mean_tss > highest_val_tss:
            highest_val_tss = mean_tss
            best_feat_count = k

    print("\n--------------------------------------------------")
    print(
        f"OPTIMAL FEATURE COUNT (Selected via Validation TSS): Top {best_feat_count} Features"
    )
    print("--------------------------------------------------")

    # --- FINAL EVALUATION ON UNTOUCHED TEST SET ---
    selector_final = SelectKBest(
        score_func=f_classif, k=min(best_feat_count, X_tr.shape[1])
    )
    selector_final.fit(X_tr, y_tr)
    final_cols = X_tr.columns[selector_final.get_support()]

    X_tr_final = X_tr[final_cols]
    X_te_final = X_te[final_cols]

    test_accs = []
    test_tss_scores = []

    for seed in range(n_runs):
        rf_final = RandomForestClassifier(
            n_estimators=100, class_weight="balanced", random_state=42 + seed
        )
        rf_final.fit(X_tr_final, y_tr)

        y_test_pred = rf_final.predict(X_te_final)

        acc_t = accuracy_score(y_te, y_test_pred)

        cm_t = confusion_matrix(y_te, y_test_pred)
        if cm_t.shape == (2, 2):
            tn, fp, fn, tp = cm_t.ravel()
            tpr_t = tp / (tp + fn) if (tp + fn) > 0 else 0
            fpr_t = fp / (fp + tn) if (fp + tn) > 0 else 0
            tss_t = tpr_t - fpr_t
        else:
            tss_t = 0.0

        test_accs.append(acc_t)
        test_tss_scores.append(tss_t)

    final_mean_acc = np.mean(test_accs)
    final_std_acc = np.std(test_accs)
    final_mean_tss = np.mean(test_tss_scores)
    final_std_tss = np.std(test_tss_scores)

    print(
        f"\nFINAL TEST SET EVALUATION (Top {best_feat_count} Features, {n_runs} Runs):"
    )
    print(f"  Test Accuracy: {final_mean_acc:.4f} ± {final_std_acc:.4f}")
    print(f"  Test TSS:      {final_mean_tss:.4f} ± {final_std_tss:.4f}")
    print("==================================================\n")

    return pd.DataFrame(val_summary), (
        final_mean_acc,
        final_std_acc,
        final_mean_tss,
        final_std_tss,
    )


# =====================================================================
# 4. EXECUTION FOR SHARP DATASET
# =====================================================================
print("--- PREPARING SHARP DATASET ---")

sharp_dataset = pd.concat(
    [
        sharp_positive_classes,
        sharp_negative_classes.sample(n=500, random_state=42),
    ],
    ignore_index=True,
)

sharp_features = [
    "HARPNUM",
    "NOAA_ARS",
    "T_REC",
    "USFLUX",
    "TOTPOT",
    "TOTUSJZ",
    "SAVNCPP",
    "AREA_ACR",
    "MEANPOT",
    "R_VALUE",
    "TOTUSJH",
]

# Group Split using HARPNUM
X_tr_raw_s, X_va_raw_s, X_te_raw_s, y_tr_s, y_va_s, y_te_s = (
    group_train_val_test_split(
        df=sharp_dataset,
        group_col="HARPNUM",
        target_col="FLARE_LABEL",
        feature_cols=sharp_features,
        test_size=0.15,
        val_size=0.15,
    )
)

# Preprocess SHARP (Train Statistics Only)
X_tr_s, X_va_s, X_te_s = prepare_train_val_test(
    X_tr_raw_s, X_va_raw_s, X_te_raw_s
)

# Run SHARP Experiment Sweep
sharp_val_df, sharp_test_results = run_feature_count_experiments(
    X_tr=X_tr_s,
    X_va=X_va_s,
    X_te=X_te_s,
    y_tr=y_tr_s,
    y_va=y_va_s,
    y_te=y_te_s,
    dataset_name="SHARP",
    n_runs=10,
)


# =====================================================================
# 5. EXECUTION FOR SMARP DATASET
# =====================================================================
print("--- PREPARING SMARP DATASET ---")

smarp_dataset = pd.concat(
    [
        smarp_positive_classes,
        smarp_negative_classes.sample(n=500, random_state=42),
    ],
    ignore_index=True,
)

smarp_features = ["TARPNUM", "NOAA_ARS", "T_REC",
    "AREA_SPT",
    "AREA_ACR",
    "NPIX_SPT",
    "USFLUXL",
    "R_VALUE",
    "SIZE_ACR",
    "MTOT",
    "MNEG_TOT",
    "MPOS_TOT",
    "MSTDEV",]

# Group Split using TARPNUM
X_tr_raw_sm, X_va_raw_sm, X_te_raw_sm, y_tr_sm, y_va_sm, y_te_sm = (
    group_train_val_test_split(
        df=smarp_dataset,
        group_col="TARPNUM",
        target_col="FLARE_LABEL",
        feature_cols=smarp_features,
        test_size=0.15,
        val_size=0.15,
    )
)

# Preprocess SMARP (Train Statistics Only)
X_tr_sm, X_va_sm, X_te_sm = prepare_train_val_test(
    X_tr_raw_sm, X_va_raw_sm, X_te_raw_sm
)

# Run SMARP Experiment Sweep
smarp_val_df, smarp_test_results = run_feature_count_experiments(
    X_tr=X_tr_sm,
    X_va=X_va_sm,
    X_te=X_te_sm,
    y_tr=y_tr_sm,
    y_va=y_va_sm,
    y_te=y_te_sm,
    dataset_name="SMARP",
    n_runs=10,
)

--- PREPARING SHARP DATASET ---
  FEATURE COUNT EXPERIMENTS FOR SHARP DATASET

--- Top 8 Selected Features (ANOVA F-Score) ---
  1. USFLUX: 436.2199
  2. TOTUSJZ: 402.7520
  3. TOTUSJH: 367.8532
  4. AREA_ACR: 350.7904
  5. SAVNCPP: 314.7196
  6. R_VALUE: 300.1568
  7. TOTPOT: 289.4730
  8. MEANPOT: 266.4623

[Top 8 Features] Validation Performance over 10 runs:
  Mean Accuracy: 0.9057 ± 0.0066
  Mean TSS:      0.7786 ± 0.0146
  Selected Features: ['USFLUX', 'TOTPOT', 'TOTUSJZ', 'SAVNCPP', 'AREA_ACR', 'MEANPOT', 'R_VALUE', 'TOTUSJH']

--- Top 10 Selected Features (ANOVA F-Score) ---
  1. USFLUX: 436.2199
  2. TOTUSJZ: 402.7520
  3. TOTUSJH: 367.8532
  4. AREA_ACR: 350.7904
  5. SAVNCPP: 314.7196
  6. R_VALUE: 300.1568
  7. TOTPOT: 289.4730
  8. MEANPOT: 266.4623
  9. HARPNUM: 26.1150
  10. NOAA_ARS: 13.1804

[Top 10 Features] Validation Performance over 10 runs:
  Mean Accuracy: 0.9115 ± 0.0061
  Mean TSS:      0.7739 ± 0.0173
  Selected Features: ['HARPNUM', 'NOAA_ARS', 'USFLUX', 'TOT

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

def evaluate_baseline_05_threshold(X_tr, X_te, y_tr, y_te, dataset_name="SHARP", n_runs=10):
    print(f"==================================================")
    print(f"  BASELINE EVALUATION (Threshold = 0.50) FOR {dataset_name}")
    print(f"==================================================")

    accs = []
    tprs = []
    fprs = []
    tss_scores = []

    for seed in range(n_runs):
        rf = RandomForestClassifier(
            n_estimators=100, class_weight="balanced", random_state=42 + seed
        )
        rf.fit(X_tr, y_tr)

        # Predict using default 0.5 probability threshold
        y_pred = rf.predict(X_te)

        acc = accuracy_score(y_te, y_pred)
        cm = confusion_matrix(y_te, y_pred)

        if cm.shape == (2, 2):
            tn, fp, fn, tp = cm.ravel()
            tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
            fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
            tss = tpr - fpr
        else:
            tpr, fpr, tss = 0.0, 0.0, 0.0

        accs.append(acc)
        tprs.append(tpr)
        fprs.append(fpr)
        tss_scores.append(tss)

    print(f"Results across {n_runs} runs:")
    print(f"  Test Accuracy:   {np.mean(accs):.4f} ± {np.std(accs):.4f}")
    print(f"  Test TPR (Recall): {np.mean(tprs):.4f} ± {np.std(tprs):.4f}")
    print(f"  Test FPR:        {np.mean(fprs):.4f} ± {np.std(fprs):.4f}")
    print(f"  Test TSS:        {np.mean(tss_scores):.4f} ± {np.std(tss_scores):.4f}")
    print("==================================================\n")

# --- Execute Baseline for SHARP ---
evaluate_baseline_05_threshold(
    X_tr=X_tr_s, X_te=X_te_s, y_tr=y_tr_s, y_te=y_te_s, dataset_name="SHARP", n_runs=10
)

# --- Execute Baseline for SMARP ---
evaluate_baseline_05_threshold(
    X_tr=X_tr_sm, X_te=X_te_sm, y_tr=y_tr_sm, y_te=y_te_sm, dataset_name="SMARP", n_runs=10
)

  BASELINE EVALUATION (Threshold = 0.50) FOR SHARP
Results across 10 runs:
  Test Accuracy:   0.8753 ± 0.0072
  Test TPR (Recall): 0.6360 ± 0.0280
  Test FPR:        0.0417 ± 0.0000
  Test TSS:        0.5943 ± 0.0280

  BASELINE EVALUATION (Threshold = 0.50) FOR SMARP
Results across 10 runs:
  Test Accuracy:   0.8722 ± 0.0075
  Test TPR (Recall): 0.7059 ± 0.0000
  Test FPR:        0.0890 ± 0.0092
  Test TSS:        0.6168 ± 0.0092

